# Open Notebook directly in Google Colab:

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/Oxford_multi_agent_investment_analysis/blob/main/Oxford_LangGraph_multi_agents_investment_analysis.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Create a Team of Agents to do Investment Analysis

<img src="https://drive.google.com/uc?export=view&id=1ohtNZSfv36WAMLhO50yHf9nmZdqzyBI4" alt="Graph" width="900">


This notebook draws inspiration from the paper [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155) by Wu et al., and the examples from [LangGraph](https://github.com/langchain-ai/langgraph/tree/main/examples/multi_agent). In this notebook, you will construct a Market Research Team and learn the following:

- How to define tools for agents. You will use the following tools:

  - [Exa](https://exa.ai/search), after account login, get your [API key here](https://docs.exa.ai/reference/getting-started-with-python). To find the exact content you're looking for on the web using embeddings-based search.  
  - [SerpApi here](https://serpapi.com/), after account login, get your [API key](https://serpapi.com/dashboard) to do look for existing patents.
  - [Python REPL](https://python.langchain.com/docs/integrations/tools/python/), please note that Python REPL can execute arbitrary code on the host machine (e.g., delete files, make network requests). Use with caution.
  - Tools to access and write to a `.txt` file and create a plot of historical prices.
- How to define utilities to help create the graph.
- How to create a team supervisor and the team of agents.


</br>

## LangGraph
You will be using [LangGraph](https://langchain-ai.github.io/langgraph/) for this. LangGraph is a library designed for building stateful, multi-actor applications with LLMs, facilitating the creation of agent and multi-agent workflows.

LangGraph supports the definition of flows that involve `cycles`, which are crucial for most __agentic architectures__. It is a low-level framework and provides you, therefore, fine-grained control over both the flow and state of your application.

LangGraph is inspired by [Pregel](https://research.google/pubs/pub37252/) and [Apache Beam](https://beam.apache.org/). The public interface draws inspiration from [NetworkX](https://networkx.org/documentation/latest/).

</br>

### Key Features

- __Cycles and Branching__: Implement loops and conditionals in your apps.
- __Persistence__: Automatically save state after each step in the graph. Pause and resume the graph execution at any point to support error recovery, human-in-the-loop workflows, time travel and more.
- __Human-in-the-Loop__: Interrupt graph execution to approve or edit next action planned by the agent.
- __Streaming Support__: Stream outputs as they are produced by each node (including token streaming).
- __Integration with LangChain__: LangGraph integrates seamlessly with [LangChain](https://www.langchain.com/) and [LangSmith](https://docs.smith.langchain.com/), but does not require them.

# Install Dependencies

In [1]:
!pip install -q langchain \
                langgraph==0.4.3 \
                langchain_community \
                langchain_experimental \
                langchain_openai \
                python-dotenv==1.0.1 \
                google-search-results \
                exa-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 22.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflic

In [2]:
! pip -q install yahooquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.5 MB/s eta 0:00:00


In [3]:
%%capture
!pip install openai==1.55.3 httpx==0.27.2 --force-reinstall --quiet

# Imports

In [4]:
# Basic imports
import os
import functools
import operator
from tempfile import TemporaryDirectory
from pathlib import Path
from typing import Annotated, List, Dict, Optional
from typing_extensions import TypedDict
from exa_py import Exa
import json
from yahooquery import Ticker
import pandas as pd
import numpy as np
from datetime import datetime



# LangChain & LangGraph imports
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser
from langchain_openai import ChatOpenAI
from langchain_openai.chat_models import ChatOpenAI
from langgraph.graph import END, StateGraph, START
from langchain_community.utilities import SerpAPIWrapper
from langchain_experimental.utilities import PythonREPL


# Display the graph
from IPython.display import Image, display

# API access
from dotenv import load_dotenv
load_dotenv()

True

# API Setup

In [5]:
# --- API Key Setup ---
# Option 1 (preferred): create a `.env` file in your project folder with:
# OPENAI_API_KEY=your_openai_key_here
# TAVILY_API_KEY=your_tavily_key_here
#
# Option 2: set it directly in the notebook with magic:
# %env OPENAI_API_KEY=your_openai_key_here
# %env TAVILY_API_KEY=your_tavily_key_here

from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

# Setup .env file via the terminal
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
serp_api_key = os.getenv("SERPAPI_API_KEY")
exa_api_key = os.getenv("EXA_API_KEY")

# Fallback: ask if still missing
if not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    OPENAI_API_KEY = input("Enter your OPENAI_API_KEY: ").strip()

if not TAVILY_API_KEY:
    print("⚠️ TAVILY_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    TAVILY_API_KEY = input("Enter your TAVILY_API_KEY: ").strip()

if not serp_api_key:
    print("⚠️ serp_api_key not found. You can set it with `%env` in the notebook or enter it below.")
    serp_api_key = input("Enter your TAVILY_API_KEY: ").strip()

if not exa_api_key:
    print("⚠️ exa_api_key not found. You can set it with `%env` in the notebook or enter it below.")
    exa_api_key = input("Enter your TAVILY_API_KEY: ").strip()

print("✅ API keys loaded successfully!")


✅ API keys loaded successfully!


# Define a Persistent Working Directory

In [6]:
# Define a persistent working directory
WORKING_DIRECTORY = Path("/content/working_directory")

# Ensure the working directory exists
if not WORKING_DIRECTORY.exists():
    WORKING_DIRECTORY.mkdir(parents=True)
    print(f"Created working directory: {WORKING_DIRECTORY}")
else:
    print(f"Working directory already exists: {WORKING_DIRECTORY}")


Created working directory: /content/working_directory


# Create Tools

Here you will create the tools for your agents. You can extend your agents [toolbox](https://python.langchain.com/v0.2/docs/concepts/#tools) as you wish.
The `@tool` decorator is the simpliest way of defining a custom tool. Note that you __need__ to define a `docstring` as the tool's description.

</br>

__Note of caution__:

The document writing and file-access tools gives your agents access to your file-system, this can be unsafe. Also, the `PythonREPL()` executes your code locally, this can introduce risks if you are not sandboxing it.


In [7]:
!pip install yahooquery

In [8]:
########## Websearch Tool ##########
@tool("finance_search")
def finance_search(query: str) -> str:
    """Search with Google SERP API by a query to Search Google for the ticker symbol for yahoo finance for a given {asset}."""
    params = {
        "engine": "google",
        "gl": "us",
        "hl": "en",
        }
    finance_search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return finance_search.run(query)



########## Coding Tool ##########
repl = PythonREPL()
# Warning: This executes code locally, which can be unsafe when not sandboxed

@tool
def python_repl(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    result_str = f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"
    return (
        result_str + "\n\nIf you have completed all tasks, respond with FINAL ANSWER."
    )


########## Analysis Tools ##########

@tool("yf_fundamental_analysis")
def yf_fundamental_analysis(ticker: str):
    """
        Perform a comprehensive fundamental analysis on the given stock symbol {stock}.

        Args:
            stock_symbol (str): The stock symbol to analyze.

        Returns:
            dict: A dictionary with the detailed fundamental analysis results.
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Data processing
        financials = stock.financials.infer_objects(copy=False)
        balance_sheet = stock.balance_sheet.infer_objects(copy=False)
        cash_flow = stock.cashflow.infer_objects(copy=False)

        # Fill missing values
        financials = financials.ffill()
        balance_sheet = balance_sheet.ffill()
        cash_flow = cash_flow.ffill()

        # Key Ratios and Metrics
        ratios = {
            "P/E Ratio": info.get('trailingPE'),
            "Forward P/E": info.get('forwardPE'),
            "P/B Ratio": info.get('priceToBook'),
            "P/S Ratio": info.get('priceToSalesTrailing12Months'),
            "PEG Ratio": info.get('pegRatio'),
            "Debt to Equity": info.get('debtToEquity'),
            "Current Ratio": info.get('currentRatio'),
            "Quick Ratio": info.get('quickRatio'),
            "ROE": info.get('returnOnEquity'),
            "ROA": info.get('returnOnAssets'),
            "ROIC": info.get('returnOnCapital'),
            "Gross Margin": info.get('grossMargins'),
            "Operating Margin": info.get('operatingMargins'),
            "Net Profit Margin": info.get('profitMargins'),
            "Dividend Yield": info.get('dividendYield'),
            "Payout Ratio": info.get('payoutRatio'),
        }

        # Growth Rates
        revenue = financials.loc['Total Revenue']
        net_income = financials.loc['Net Income']
        revenue_growth = revenue.pct_change(periods=-1).iloc[0] if len(revenue) > 1 else None
        net_income_growth = net_income.pct_change(periods=-1).iloc[0] if len(net_income) > 1 else None

        growth_rates = {
            "Revenue Growth (YoY)": revenue_growth,
            "Net Income Growth (YoY)": net_income_growth,
        }

        # Valuation
        market_cap = info.get('marketCap')
        enterprise_value = info.get('enterpriseValue')

        valuation = {
            "Market Cap": market_cap,
            "Enterprise Value": enterprise_value,
            "EV/EBITDA": info.get('enterpriseToEbitda'),
            "EV/Revenue": info.get('enterpriseToRevenue'),
        }

        # Future Estimates
        estimates = {
            "Next Year EPS Estimate": info.get('forwardEps'),
            "Next Year Revenue Estimate": info.get('revenueEstimates', {}).get('avg'),
            "Long-term Growth Rate": info.get('longTermPotentialGrowthRate'),
        }

        # Simple DCF Valuation (very basic)
        free_cash_flow = cash_flow.loc['Free Cash Flow'].iloc[0] if 'Free Cash Flow' in cash_flow.index else None
        wacc = 0.1  # Assumed Weighted Average Cost of Capital
        growth_rate = info.get('longTermPotentialGrowthRate', 0.03)

        def simple_dcf(fcf, growth_rate, wacc, years=5):
            if fcf is None or growth_rate is None:
                return None
            terminal_value = fcf * (1 + growth_rate) / (wacc - growth_rate)
            dcf_value = sum([fcf * (1 + growth_rate) ** i / (1 + wacc) ** i for i in range(1, years + 1)])
            dcf_value += terminal_value / (1 + wacc) ** years
            return dcf_value

        dcf_value = simple_dcf(free_cash_flow, growth_rate, wacc)

        # Prepare the results
        analysis = {
            "Company Name": info.get('longName'),
            "Sector": info.get('sector'),
            "Industry": info.get('industry'),
            "Key Ratios": ratios,
            "Growth Rates": growth_rates,
            "Valuation Metrics": valuation,
            "Future Estimates": estimates,
            "Simple DCF Valuation": dcf_value,
            "Last Updated": datetime.fromtimestamp(info.get('lastFiscalYearEnd', 0)).strftime('%Y-%m-%d'),
            "Data Retrieval Date": datetime.now().strftime('%Y-%m-%d'),
        }

        # Add interpretations
        interpretations = {
            "P/E Ratio": "High P/E might indicate overvaluation or high growth expectations" if ratios.get('P/E Ratio', 0) > 20 else "Low P/E might indicate undervaluation or low growth expectations",
            "Debt to Equity": "High leverage" if ratios.get('Debt to Equity', 0) > 2 else "Conservative capital structure",
            "ROE": "Strong returns" if ratios.get('ROE', 0) > 0.15 else "Potential profitability issues",
            "Revenue Growth": "Strong growth" if growth_rates.get('Revenue Growth (YoY)', 0) > 0.1 else "Slowing growth",
        }

        analysis["Interpretations"] = interpretations

        return analysis

    except Exception as e:
        return f"An error occurred during the analysis: {str(e)}"


def compute_tech_indicators(data, value):
    # Compute 10 and 30 days Moving Averages
    data['ma10'] = data[value].rolling(window=10).mean()
    data['ma30'] = data[value].rolling(window=30).mean()

    # Create MACD with shorter windows
    data['13ema'] = data[value].ewm(span=13).mean()
    data['6ema'] = data[value].ewm(span=6).mean()
    data['MACD'] = data['6ema'] - data['13ema']

    # Ensure `MACD` calculation is done before creating `MACD_Signal`
    data['MACD_Signal'] = create_MACD_signal(data['MACD'])

    # Create Exponential Moving Average (shorter window)
    data['ema'] = data[value].ewm(com=0.3).mean()

    # Create Momentum
    data['momentum'] = (data[value] / 100) - 1

    # RSI
    delta = data[value].diff()
    gain = delta.where(delta > 0, 0).rolling(window=7).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=7).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    return data

def create_MACD_signal(macd_series):
    macd_sign = np.sign(macd_series)
    macd_shifted_sign = macd_sign.shift(1)
    return macd_sign * (macd_sign != macd_shifted_sign)

def generate_MA_crossing(data, value, s_window=20, l_window=50):
    data['short_MA'] = data[value].rolling(window=s_window).mean()
    data['long_MA'] = data[value].rolling(window=l_window).mean()
    data['short_MA-long_MA'] = data['short_MA'] - data['long_MA']
    data['Signal'] = np.where(data['short_MA-long_MA'] > 0, 1, 0)
    data['Signal'] = np.where(data['short_MA-long_MA'] < 0, -1, data['Signal'])
    data['Trading'] = np.sign(data['Signal'] - data['Signal'].shift(1))
    return data



@tool("yf_tech_analysis")
def yf_tech_analysis(stock_symbol: str, period: str = "1y"):
    """
    Perform a comprehensive technical analysis on the given stock symbol using yahooquery.

    Args:
        stock_symbol (str): The asset symbol to analyze.
        period (str, optional): The time period for analysis. Defaults to "1y".

    Returns:
        analysis_results_df: pandas dataframe, interpretation: dictionary
    """
    # Map period to yahooquery format
    period_mapping = {
        "1y": "1y",
        "6mo": "6mo",
        "3mo": "3mo",
        "1mo": "1mo",
        "5d": "5d",
        "1d": "1d"
    }

    if period not in period_mapping:
        raise ValueError(f"Unsupported period: {period}")

    # Fetch data
    ticker = Ticker(stock_symbol)
    data = ticker.history(period=period_mapping[period])
    if isinstance(data, dict) or data.empty:
        raise ValueError(f"No data returned for symbol: {stock_symbol}")

    # Handle multi-index format from yahooquery
    if isinstance(data.index, pd.MultiIndex):
        data = data.xs(stock_symbol, level=0)

    print("Initial data NaNs:\n", data.isna().sum())

    # Compute indicators
    data = compute_tech_indicators(data, 'close')
    data = generate_MA_crossing(data, 'close')

    # Forward fill rolling windows
    data[['ma10', 'ma30', 'short_MA', 'long_MA']] = data[['ma10', 'ma30', 'short_MA', 'long_MA']].ffill()
    data = data.dropna()

    if data.empty:
        raise ValueError("Insufficient data after calculations. Increase the period or decrease indicator windows.")

    # Create analysis results
    analysis_results_df = pd.DataFrame({
        'Current_Price': [data['close'].iloc[-1]],
        '10_MA': [data['ma10'].iloc[-1]],
        '30_MA': [data['ma30'].iloc[-1]],
        'Short_MA': [data['short_MA'].iloc[-1]],
        'Long_MA': [data['long_MA'].iloc[-1]],
        '6_EMA': [data['6ema'].iloc[-1]],
        '13_EMA': [data['13ema'].iloc[-1]],
        'EMA': [data['ema'].iloc[-1]],
        'MACD_Value': [data['MACD'].iloc[-1]],
        'MACD_Signal': [data['MACD_Signal'].iloc[-1]],
        'RSI': [data['RSI'].iloc[-1]],
        'Momentum': [data['momentum'].iloc[-1]],
        'MA_Crossing_Signal': [data['Signal'].iloc[-1]],
        'Trading_Action': [data['Trading'].iloc[-1]]
    })

    # Extract scalars for interpretation
    latest_close = data['close'].iloc[-1]
    latest_ma30 = data['ma30'].iloc[-1]
    latest_macd = data['MACD'].iloc[-1]
    latest_rsi = data['RSI'].iloc[-1]

    interpretation = {
        'Trend': 'Bullish' if latest_close > latest_ma30 else 'Bearish',
        'RSI': 'Overbought' if latest_rsi > 70 else ('Oversold' if latest_rsi < 30 else 'Neutral'),
        'MACD': 'Bullish' if latest_macd > 0 else 'Bearish'
    }

    return analysis_results_df, interpretation



########## News Sentiment Tools ##########

@tool("news_search")
def news_search(user_prompt: str) -> str:
    """
    Generate a search query based on the user's prompt, fetch relevant news articles using Exa, and summarize the content.

    Args:
        user_prompt (str): The user's question or topic to search for.

    Returns:
        str: A summary of the news articles with URLs.
    """
    import openai
    from exa_py import Exa
    from datetime import datetime, timedelta

    openai.api_key = OPENAI_API_KEY
    exa = Exa(exa_api_key)

    SYSTEM_MESSAGE = "You are a helpful assistant that generates search queries based on user questions. Only generate one search query."

    completion = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_prompt},
        ],
    )
    search_query = completion.choices[0].message.content.strip()


    one_week_ago = (datetime.now() - timedelta(days=7))
    date_cutoff = one_week_ago.strftime("%Y-%m-%d")
    search_response = exa.search_and_contents(
        search_query, use_autoprompt=True, start_published_date=date_cutoff
    )

    results = search_response.results
    summaries = []
    for result_item in results[:5]:  # Limit to top 5 results for brevity
        summary_completion = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that briefly summarizes the content of a webpage."},
                {"role": "user", "content": result_item.text},
            ],
        )
        summary = summary_completion.choices[0].message.content.strip()
        summaries.append(f"Title: {result_item.title}\nURL: {result_item.url}\nSummary: {summary}\n")

    # Combine summaries into a single string
    return "\n\n".join(summaries)


@tool("sentiment_analysis")
def sentiment_analysis(text: str) -> str:
    """Perform sentiment analysis on financial news."""
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch

    # Load the model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
    model = AutoModelForSequenceClassification.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Get sentiment prediction
    outputs = model(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    sentiment = torch.argmax(predictions, dim=1).item()

    labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
    return f"Sentiment: {labels[sentiment]}, Confidence: {predictions[0][sentiment].item():.2f}"





########## Document Tools ##########
@tool
def create_outline(
    points: Annotated[List[str], "List of main points or sections."],
    file_name: Annotated[str, "File path to save the outline."],
) -> Annotated[str, "Path of the saved outline file."]:
    """Create and save an outline."""
    with (WORKING_DIRECTORY / file_name).open("w") as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"


@tool
def read_document(
    file_name: Annotated[str, "File path to save the document."],
    start: Annotated[Optional[int], "The start line. Default is 0"] = None,
    end: Annotated[Optional[int], "The end line. Default is None"] = None,
) -> str:
    """Read the specified document."""
    with (WORKING_DIRECTORY / file_name).open("r") as file:
        lines = file.readlines()
    if start is not None:
        start = 0
    return "\n".join(lines[start:end])


@tool
def write_document(
    content: Annotated[str, "Text content to be written into the document."],
    file_name: Annotated[str, "File path to save the document."],
) -> Annotated[str, "Path of the saved document file."]:
    """Create and save a text document."""
    with (WORKING_DIRECTORY / file_name).open("w") as file:
        file.write(content)
    return f"Document saved to {file_name}"


@tool
def edit_document(
    file_name: Annotated[str, "Path of the document to be edited."],
    inserts: Annotated[
        Dict[int, str],
        "Dictionary where key is the line number (1-indexed) and value is the text to be inserted at that line.",
    ],
) -> Annotated[str, "Path of the edited document file."]:
    """Edit a document by inserting text at specific line numbers."""

    with (WORKING_DIRECTORY / file_name).open("r") as file:
        lines = file.readlines()

    sorted_inserts = sorted(inserts.items())

    for line_number, text in sorted_inserts:
        if 1 <= line_number <= len(lines) + 1:
            lines.insert(line_number - 1, text + "\n")
        else:
            return f"Error: Line number {line_number} is out of range."

    with (WORKING_DIRECTORY / file_name).open("w") as file:
        file.writelines(lines)

    return f"Document edited and saved to {file_name}"


# Create Agent and Team Supervisor

In [9]:
def create_agent(
    llm: ChatOpenAI,
    tools: list,
    system_prompt: str,
) -> str:
    """Create a function-calling agent and add it to the graph."""
    system_prompt += "\nWork autonomously according to your specialty, using the tools available to you."
    " Do not ask for clarification."
    " Your other team members (and other teams) will collaborate with you with their own specialties."
    " You are chosen for a reason! You are one of the following team members: {team_members}."
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                system_prompt,
            ),
            MessagesPlaceholder(variable_name="messages"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ]
    )
    agent = create_openai_functions_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, handle_parsing_errors=True)
    return executor


def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {"messages": [HumanMessage(content=result["output"], name=name)]}


def create_team_supervisor(llm: ChatOpenAI, system_prompt, members) -> str:
    """An LLM-based router."""
    options = ["FINISH"] + members
    function_def = {
        "name": "route",
        "description": "Select the next role.",
        "parameters": {
            "title": "routeSchema",
            "type": "object",
            "properties": {
                "next": {
                    "title": "Next",
                    "anyOf": [
                        {"enum": options},
                    ],
                },
            },
            "required": ["next"],
        },
    }
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="messages"),
            (
                "system",
                "Given the conversation above, who should act next?"
                " Or should we FINISH? Select one of: {options}",
            ),
        ]
    ).partial(options=str(options), team_members=", ".join(members))
    return (
        prompt
        | llm.bind_functions(functions=[function_def], function_call="route")
        | JsonOutputFunctionsParser()
    )

# Create the Agents and Graph

Here you create the agents you can also combine different LLMs to do your tasks. This might be beneficial because for specific tasks some LLMs might perform better than others or be less costly than others.

First you have to define the `graph state`. And then you can create your agents which can access the tools created before.

**State**: A shared data structure that represents the current snapshot of an application. It can be any Python type, but is typically a TypedDict or Pydantic BaseModel.


## Define Graph State

In [10]:
# Define team graph state
class FinanceTeamState(TypedDict):
    # This tracks the team's conversation internally
    messages: Annotated[List[BaseMessage], operator.add]
    # This provides each worker with context on the others' skill sets
    team_members: str
    # This is how the supervisor tells langgraph who has to work next
    next: str
    # This tracks the shared directory state
    current_files: str

Next, you have a helper function that will run before each worker agent begins their task. This function ensures that the agents are more aware of the current state of the working directory.


## Helper Function for Dictory Access

In [11]:
def prelude(state):
    """
    A helper function that prepares the state for each worker agent by ensuring the working directory exists
    and listing the files present in it.

    Args:
        state (dict): The current state to be updated with information about the files in the working directory.

    Returns:
        dict: The updated state with a key "current_files" containing a message about the files in the directory.
              If no files are present, the message will indicate that no files have been written.
    """

    written_files = []
    if not WORKING_DIRECTORY.exists():
        WORKING_DIRECTORY.mkdir(parents=True)
    try:
        written_files = [
            f.relative_to(WORKING_DIRECTORY) for f in WORKING_DIRECTORY.rglob("*")
            if f.is_file()
        ]
    except Exception as e:
        print(f"Error reading files: {e}")
    if not written_files:
        return {**state, "current_files": "No files written."}
    return {
        **state,
        "current_files": "\nBelow are files your team has written to the directory:\n"
        + "\n".join([f" - {f}" for f in written_files]),
    }

# Init LLM

In [12]:
######## Initialize the LLM ########
llm = ChatOpenAI(model="gpt-4.1")

# Define the Agents

__ATT:__ We to escape the curly braces, that is, we have to wrap the placeholder in double curly braces `{{asset}}` so it’s not mistaken for a variable in ChatPromptTemplate. With this, the stock will be treated as part of the prompt text rather than requiring an actual variable input.

In [13]:
ticker_search_agent = create_agent(
    llm,
    [finance_search],
    """You just look up the correct symbol for futures for a given {{asset}} so it can be properly retrieved from yahoo finance.""",
)
ticker_search_node = functools.partial(agent_node, agent=ticker_search_agent, name="TickerSearch")

news_search_agent = create_agent(
    llm,
    [news_search],
    """You are a financial analyst specializing in generating search queries
    and finding recent news articles on a given {{asset}}. Given the user's {{asset}} you create a topic, and perform a news search and summarize the most relevant articles.""",
)
news_search_node = functools.partial(agent_node, agent=news_search_agent, name="NewsSearch")


sentiment_agent = create_agent(
    llm,
    [sentiment_analysis],
    """You are a financial analyst specializing in sentiment analysis of news articles. Analyze the sentiment of the provided summary.""",
)
sentiment_node = functools.partial(agent_node, agent=sentiment_agent, name="SentimentAnalysis")


tech_analysis_agent = create_agent(
    llm,
    [yf_tech_analysis],
    """You are Chartered Market Technician (CMT) with 25 years of experience,
    you have a keen eye for market trends and patterns on {{asset}} and this {{date}}.""",
)
tech_analysis_node = functools.partial(agent_node, agent=tech_analysis_agent, name="TechnicalAnalysis")

fundamental_analysis_agent = create_agent(
    llm,
    [yf_fundamental_analysis],
    """With a CFA charter and over 20 years of experience in investing,
    you dissect financial statements and identify key value drivers on {{stock_symbol}} and this {{date}}.""",
)
fundamental_analysis_node = functools.partial(agent_node, agent=fundamental_analysis_agent, name="FinanceAnalysis")


chart_agent = create_agent(
    llm,
    [python_repl],
    """You are a Quant Developer and can write code to plot any charts requested.""",
)
chart_node = functools.partial(agent_node, agent=chart_agent, name="ChartGenerator")

doc_writer_agent = create_agent(
    llm,
    [write_document, edit_document, read_document],
    """You are a Chief Investment Strategist, who synthesize all analyses to create
    a definitive investment report on {{stock_symbol}}.
    \n""",
)

# Injects current directory working state before each call
context_aware_doc_writer_agent = prelude | doc_writer_agent
doc_writing_node = functools.partial(
    agent_node, agent=context_aware_doc_writer_agent, name="DocWriter"
)

investment_analysis_supervisor = create_team_supervisor(
    llm,
    "You are a supervisor tasked with managing a conversation between the"
    " following workers:  {team_members}. Given the following user request,"
    " respond with the worker to act next. Each worker will perform a"
    " task and respond with their results and status. When finished,"
    " respond with FINISH.",
    ["TickerSearch","NewsSearch", "SentimentAnalysis", "TechnicalAnalysis", "FinanceAnalysis", "ChartGenerator" ,"DocWriter"],
)

/tmp/ipython-input-3422690559.py:64: LangChainDeprecationWarning: The method `BaseChatOpenAI.bind_functions` was deprecated in langchain-openai 0.2.1 and will be removed in 1.0.0. Use :meth:`~langchain_openai.chat_models.base.ChatOpenAI.bind_tools` instead.
  | llm.bind_functions(functions=[function_def], function_call="route")


## Create the Graph

- **Nodes**: Functions that encode the logic of the agents. They receive the current state as input, perform some computation or side-effect, and return an updated state.
- **Edges**: Functions that determine which node to execute next based on the current state. They can be conditional branches or fixed transitions.

In [14]:
# Create the graph
financial_graph = StateGraph(FinanceTeamState)
financial_graph.add_node("DocWriter", doc_writing_node)
financial_graph.add_node("TickerSearch", ticker_search_node)
financial_graph.add_node("NewsSearch", news_search_node)
financial_graph.add_node("SentimentAnalysis", sentiment_node)
financial_graph.add_node("TechnicalAnalysis", tech_analysis_node)
financial_graph.add_node("FinanceAnalysis", fundamental_analysis_node)
financial_graph.add_node("ChartGenerator", chart_node)
financial_graph.add_node("supervisor", investment_analysis_supervisor)

# Add the edges
financial_graph.add_edge("DocWriter", "supervisor")
financial_graph.add_edge("TickerSearch", "supervisor")
financial_graph.add_edge("NewsSearch", "supervisor")
financial_graph.add_edge("SentimentAnalysis", "supervisor")
financial_graph.add_edge("TechnicalAnalysis", "supervisor")
financial_graph.add_edge("FinanceAnalysis", "supervisor")
financial_graph.add_edge("ChartGenerator", "supervisor")

# Add the edges where routing applies
financial_graph.add_conditional_edges(
    "supervisor",
    lambda x: x["next"],
    {
        "DocWriter": "DocWriter",
        "TickerSearch": "TickerSearch",
        "NewsSearch": "NewsSearch",
        "SentimentAnalysis": "SentimentAnalysis",
        "TechnicalAnalysis": "TechnicalAnalysis",
        "FinanceAnalysis": "FinanceAnalysis",
        "ChartGenerator": "ChartGenerator",
        "FINISH": END,
    },
)

financial_graph.add_edge(START, "supervisor")
chain = financial_graph.compile()


# The following functions interoperate between the top level graph state
# and the state of the research sub-graph
# this makes it so that the states of each graph don't get intermixed
def enter_chain(message: str, members: List[str]):
    results = {
        "messages": [HumanMessage(content=message)],
        "team_members": ", ".join(members),
    }
    return results

financial_chain = (
    functools.partial(enter_chain, members=financial_graph.nodes)
    | financial_graph.compile()
)


In [15]:
#display(Image(chain.get_graph().draw_mermaid_png()))

In [16]:
for s in financial_chain.stream(
    """Write an investment report on {{AAPL}} stock for this {{October, 24th 2025}}.
    First, do a technical analysis and then a fundamental anaylsis on {{AAPL}}. Also plot a
    graph of it the historical price on {{AAPL}} and save it as .png file. Finally do a sentiment
    analysis. IMPORTANT: Use the appropriate agents and tools for each task. After that, write all into an investment
    report and save it to disk as .txt file.""",
    {"recursion_limit": 100},
):
    if "__end__" not in s:
        print(s)
        print("---")


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Result Report

AAPL Investment Report – October 24, 2025

1. Technical Analysis
---------------------
Current Price (Oct 24, 2025): \$262.82
Key Moving Averages:
 - 10-Day MA: \$255.04 (uptrend)
 - 30-Day MA: \$252.43 (uptrend)
 - Short-term MA: \$255.03 vs. Long-term MA: \$244.14 → Bullish alignment
Momentum Indicators:
 - RSI: 76.02 (Overbought, warning for potential near-term pullback)
 - MACD: 2.79 (Bullish momentum)
 - Momentum: 1.63 (Uptrend)
Conclusion: Bullish technical structure, but currently overbought—investors should be mindful of pullback risk.

2. Fundamental Analysis
------------------------
Apple remains a global leader in consumer technology, boasting strong brand equity, recurring revenues, and healthy margins.
 - Revenue Growth: Historically stable, high-single-digit annual growth driven by iPhone cycles, services, and wearables.
 - Profitability: Industry-leading gross margins, aided by premium pricing and ecosystem lock-in.
 - Valuation: As of late 2025, typical P/E ratios for Apple are mid-to-high 20s, reflecting premium growth expectations.
 - Balance Sheet: Robust, with significant cash reserves and manageable debt profile.
 - Risks: Market maturity, regulatory challenges, dependency on product refresh cycles, and macroeconomic headwinds.

3. Historical Price Chart
--------------------------
![Tesla Historical Price](https://drive.google.com/uc?export=view&id=1kdLsAWPNuXBj-AYpWaaNgAgAqraQk0KN)


4. Sentiment Analysis
----------------------
Based on news headlines and analyst commentary (as of 2025):
 - Market Sentiment: Generally positive, confidence in Apple’s product pipeline and ecosystem.
 - News Tone: Upbeat, as Apple’s record services revenue and new launches fuel growth.
 - Analyst Consensus: Predominantly “Buy” or “Outperform,” target prices moderately above spot.

Investment Outlook:
--------------------
Apple remains a high-quality compounder. While technicals suggest a possible short-run correction, the long-term story is intact for buy-and-hold investors. Those increasing positions should consider scaling in to mitigate short-term overbought risk.

Report compiled on Oct 24, 2025.
